In [20]:
import requests 
from bs4 import BeautifulSoup 
import re 
import html
import html5lib
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [21]:
#для пробы получаем первую страницу сайта.
requests.get("https://www.ncbi.nlm.nih.gov/pmc/?term=machine+learning")

<Response [200]>

Метод requests.get() возвращает объект Response, который содержит большое количество различной информации о загруженной (или незагруженной) странице. В краткой форме отображается только результат выполения запроса. В нашем случае это 200, нет ошибки.
Посмотрим что результат содержит еще.

In [4]:
resp = requests.get("https://www.ncbi.nlm.nih.gov/pmc/?term=machine+learning")

In [5]:
dir(resp)

['__attrs__',
 '__bool__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__enter__',
 '__eq__',
 '__exit__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__nonzero__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_content',
 '_content_consumed',
 '_next',
 'apparent_encoding',
 'close',
 'connection',
 'content',
 'cookies',
 'elapsed',
 'encoding',
 'headers',
 'history',
 'is_permanent_redirect',
 'is_redirect',
 'iter_content',
 'iter_lines',
 'json',
 'links',
 'next',
 'ok',
 'raise_for_status',
 'raw',
 'reason',
 'request',
 'status_code',
 'text',
 'url']

#### загружаем первые 10 статей на тему machine+learning

In [22]:
# Функция для получения статей
def get_article_links(search_term, num_articles=20):
    url = f"https://www.ncbi.nlm.nih.gov/pmc/?term={search_term.replace(' ', '+')}"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Находим все статьи
    articles = soup.find_all('div', class_='rprt', limit=num_articles) #метод find_all используется для поиска всех элементов div с классом rprt, который, содержит информацию о статьях. Параметр limit=num_articles ограничивает количество найденных элементов до значения, указанного в num_articles. 

    article_links = []
    
    for article in articles:
        title_element = article.find('a', class_='view')#ищется элемент a с классом view, который содержит ссылку на статью.
        if title_element:
            article_link = title_element['href']#извлекается значение атрибута href, которое содержит URL статьи.
            article_links.append(article_link)

    return article_links


search_term = "machine learning"  #поисковый запрос
articles = get_article_links(search_term)
articles

['https://pmc.ncbi.nlm.nih.gov/articles/PMC8391798/',
 'https://pmc.ncbi.nlm.nih.gov/articles/PMC12078339/',
 'https://pmc.ncbi.nlm.nih.gov/articles/PMC12093382/',
 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11123121/',
 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9337975/',
 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11009967/',
 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11228968/',
 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11346375/',
 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7857908/',
 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9295160/',
 'https://pmc.ncbi.nlm.nih.gov/articles/PMC8391964/',
 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7433880/',
 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10575847/',
 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11926528/',
 'https://pmc.ncbi.nlm.nih.gov/articles/PMC8475847/',
 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11885302/',
 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9739236/',
 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9370746/',
 'https://pmc.ncbi.

In [23]:
#функция для получения статей
def get_article_links(search_term, num_articles=10):
    url = f"https://www.ncbi.nlm.nih.gov/pmc/?term={search_term.replace(' ', '+')}"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser') #Полученный HTML-код страницы передается в BeautifulSoup, который создает объект soup. Этот объект позволяет удобно извлекать данные из HTML-документа.

    #находим все статьи
    articles = soup.find_all('div', class_='rprt', limit=num_articles)

    article_links = []
    
    for article in articles:
        title_element = article.find('a', class_='view')
        if title_element:
            article_link = title_element['href']
            article_links.append(article_link)

    return article_links

#функция для получения содержимого статьи
def get_article_content(article_links):
    contents = []
    
    #инициализация драйвера Safari
    driver = webdriver.Safari() #создается экземпляр веб-драйвера Safari, который будет использоваться для автоматизации браузера и извлечения данных с веб-страниц.

    try:
        for link in article_links:
            #открываем страницу статьи
            driver.get(link)
            time.sleep(5)  #ждем, чтобы страница полностью загрузилась
            
            #извлечение заголовка
            title = driver.find_element(By.XPATH, '//h1').text
            
            #извлечение авторов
            authors = driver.find_elements(By.XPATH, '//div[@class="cg p"]//span[@class="name western"]')
            authors_list = [author.text for author in authors]  #список имен авторов
            unique_authors = list(set(authors_list))  #уникальные имена авторов

            #подсчет уникальных авторов
            count_ = len(unique_authors)

            #извлечение организаций для каждого автора
            organizations_list = []
            for i in range(1, count_):
                #показ скрытого элемента с организацией для текущего автора
                author_id = f'id{i}'  #id начинается с id1 для первого автора
                driver.execute_script(f"document.getElementById('{author_id}').style.display = 'block';")
                time.sleep(1)  #ждем, чтобы изменения применились
    
                #извлечение организации
                organization_element = driver.find_element(By.XPATH, f'//*[@id="{author_id}"]/div[1]') 
                organization_text = organization_element.text
                organizations_list.append(organization_text)

            
            #извлечение текста статьи
            text_elements = driver.find_elements(By.XPATH, '//*[@id="main-content"]/article/section[2]//p')
            article_text = "\n".join([text.text for text in text_elements])
            
            #сохраняем содержимое
            contents.append({
                'title': title,
                'authors': unique_authors,
                'organizations': organizations_list,
                'text': article_text
            })
    
    except Exception as e:
        print(f"Ошибка: {e}")
    
    finally:
        #закрываем драйвер
        driver.quit()
    
    return contents

search_term = "machine learning"  #поисковый запрос
articles = get_article_links(search_term)
contents = get_article_content(articles)

        
#сохранение содержимого в текстовый файл
with open('articles.txt', 'w', encoding='utf-8') as file:
    for content in contents:
        file.write(f"Title: {content['title']}\n")
        file.write(f"Authors: {', '.join(content['authors'])}\n")
        file.write(f"Organizations: {', '.join(content['organizations'])}\n")
        file.write(f"Text:\n{content['text']}\n")
        file.write("\n" + "="*50 + "\n\n")  
print("Содержимое сохранено в articles.txt")

Содержимое сохранено в articles.txt


In [12]:
#сохраним еще статьи по Кардиогенному шоку
if __name__ == "__main__":
    search_term = "cardiogenic shock"  
    articles = get_article_links(search_term)
    contents = get_article_content(articles)

    with open('articles CS.txt', 'w', encoding='utf-8') as file:
        for content in contents:
            file.write(f"Title: {content['title']}\n")
            file.write(f"Authors: {', '.join(content['authors'])}\n")
            file.write(f"Organizations: {', '.join(content['organizations'])}\n")
            file.write(f"Text:\n{content['text']}\n")
            file.write("\n" + "="*50 + "\n\n") 
    print("Содержимое сохранено в articles CS.txt")

Содержимое сохранено в articles CS.txt


In [19]:
# Функция для получения статей
def get_article_links(search_term, num_articles=100):
    base_url = "https://www.ncbi.nlm.nih.gov/pmc/?term="
    article_links = []
    page = 0  # Начинаем с первой страницы


    while len(article_links) < num_articles:
        url = f"{base_url}{search_term.replace(' ', '+')}&page={page}"
        response = requests.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')


        # Находим все статьи на текущей странице
        articles = soup.find_all('div', class_='rprt')


        if not articles:  # Если статьи не найдены, выходим из цикла
            break


        for article in articles:
            title_element = article.find('a', class_='view')
            if title_element:
                article_link = title_element['href']
                article_links.append(article_link)


            if len(article_links) >= num_articles:  # Если достигли нужного количества статей, выходим
                break


        page += 1  # Переходим на следующую страницу


    return article_links[:num_articles]  # Возвращаем только нужное количество статей


# Функция для получения содержимого статьи
def get_article_content(article_links):
    contents = []
    
    # Инициализация драйвера Safari
    driver = webdriver.Safari()

    try:
        for link in article_links:
            # Открываем страницу статьи
            driver.get(link)
            time.sleep(5)  # Ждем, чтобы страница полностью загрузилась
            
            # Извлечение заголовка
            title = driver.find_element(By.XPATH, '//h1').text
            

            
            # Извлечение текста статьи
            text_elements = driver.find_elements(By.XPATH, '//*[@id="main-content"]/article/section[2]//p')
            article_text = "\n".join([text.text for text in text_elements])
            
            # Сохраняем содержимое
            contents.append({
                'title': title,
                'text': article_text
            })
    
    except Exception as e:
        print(f"Ошибка: {e}")
    
    finally:
        # Закрываем драйвер
        driver.quit()
    
    return contents

# Основная функция
if __name__ == "__main__":
    search_term = "machine learning"  # Ваш поисковый запрос
    articles = get_article_links(search_term)
    contents = get_article_content(articles)

        
    # Сохранение содержимого в текстовый файл
    with open('articles100.txt', 'w', encoding='utf-8') as file:
        for content in contents:
            file.write(f"Title: {content['title']}\n")
            file.write(f"Text:\n{content['text']}\n")
            file.write("\n" + "="*50 + "\n\n")  # Разделитель между статьями
    print("Содержимое сохранено в articles100.txt")

Содержимое сохранено в articles100.txt
